# Ternary Phase Competition — Active Learning

A synthetic benchmark modeled on alloy phase competition. Composition $(x_A, x_B, x_C)$ lies on the 2-simplex and determines the relative stability of $N_\phi = 6$ thermodynamic phases.

Deep inside a dominant phase region the observable is typically unimodal; near phase boundaries, competing phases coexist and the conditional distribution becomes multimodal. Aleatoric noise also varies across the simplex through a separate heteroscedastic noise field, so high-noise regions need not coincide with phase boundaries.

## Step 1 — Imports and setup

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # replace with desired GPU
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import sys
from pathlib import Path

_EXAMPLE_DIR = str(Path().resolve())
_REPO_ROOT = str(Path().resolve().parents[1])
for p in (_REPO_ROOT, _EXAMPLE_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

from ternary_phases import make_phase_system, verify_phase_diagram
from active_learning_utils import (
    create_shared_ternary_data,
    get_default_mdn_config,
    make_mdn_trainer_factory,
    run_pool_based_active_learning_experiment,
    plot_loss_comparison,
    plot_acquired_on_simplex,
    plot_epistemic_vs_aleatoric,
    plot_prediction_vs_truth,
    make_single_run_figure,
)
print("JAX devices:", jax.devices())

## Step 2 — Verify the Phase Diagram

In [ ]:
system_sharp = make_phase_system(seed=4, n_phases=4, tau_G=0.08, n_proc_params=6, c_mu_scale=6.0)
_ = verify_phase_diagram(system_sharp, n_grid=150)

## Step 3 — Create the shared benchmark dataset

Because $p^*(y \mid x)$ is known analytically, we can compute the **true NLL**
on the test set — the performance floor for any model.

In [ ]:
SEED = 0
N_PHASES = 4
TAU_G = 0.08
N_PROC_PARAMS = 6
C_MU_SCALE = 6.0
SYSTEM_SEED = 12
CANDIDATE_SAMPLE_COUNT = 50_000
TEST_SAMPLE_COUNT = 2_000
INITIAL_SAMPLE_COUNT = 100

data = create_shared_ternary_data(
    seed=SEED,
    system_seed=SYSTEM_SEED,
    candidate_sample_count=CANDIDATE_SAMPLE_COUNT,
    test_sample_count=TEST_SAMPLE_COUNT,
    initial_sample_count=INITIAL_SAMPLE_COUNT,
    n_phases=N_PHASES,
    tau_G=TAU_G,
    n_proc_params=N_PROC_PARAMS,
    c_mu_scale=C_MU_SCALE,
)
print(f"pool inputs : {data['remaining_pool_inputs'].shape}")
print(f"test inputs : {data['test_data'][0].shape}")
print(f"true NLL    : {data['true_nll']:.4f}   (ensemble floor)")

## Step 4 — MDN configuration


In [ ]:
model_config = get_default_mdn_config(
    out_dim=1,
    hidden_features=64,
    depth=2,
    num_mixtures=4,       # matches true N_phi = 4
    ensemble_size=8,
)
trainer_factory = make_mdn_trainer_factory(
    model_config,
    n_iter=40_000,
    batch_size=64,
    adaptive_iters=True,
    iter_per_sample=200,
)

## Step 5 — Run the AL loop across strategies

In [ ]:
strategies = [
    'random',
    'mdn_epistemic_variance',
    'sbal_mdn_epistemic_variance',
    'mi_lb',
    'sbal_mi_lb',
    'bait',
    'coreset',
]

SBAL_TEMPERATURE = 0.3
CORESET_ENSEMBLE_MEMBER = 0
CORESET_POOL_SUBSAMPLE = None

AL_ITERS = 30
QUERY_BATCH_SIZE = 15

results = {}
for name in strategies:
    print(f'\n=== {name} ===')
    results[name] = run_pool_based_active_learning_experiment(
        trainer_factory=trainer_factory,
        shared_benchmark=data,
        acquisition_name=name,
        al_iters=AL_ITERS,
        query_batch_size=QUERY_BATCH_SIZE,
        acquisition_batch_size=256,
        sbal_temperature=SBAL_TEMPERATURE,
        coreset_ensemble_member=CORESET_ENSEMBLE_MEMBER,
        coreset_pool_subsample=CORESET_POOL_SUBSAMPLE,
    )
    print(f'  final NLL: {results[name]["final_test_nll"]:.4f}   '
          f'(true: {results[name]["true_nll"]:.4f})')

## Step 6 — Learning curves and calibration gap

In [ ]:
_ = plot_loss_comparison(results)

## Step 7 — Simplex analysis

In [ ]:
_ = plot_acquired_on_simplex(results, data)

**Phase boundary recovery.** IoU between the MDN's predicted multimodal region ($>1$ active mixture component) and the ground-truth boundary. Higher is better.

**Epistemic vs. aleatoric uncertainty** for the best-performing model.  If
the ensemble has learned the phase structure correctly, the epistemic map
should track the phase boundaries, while the aleatoric map tracks the noise
structure — two visibly different fields on the simplex.

In [ ]:
best = min(results, key=lambda k: results[k]['final_test_nll'])
print(f'Uncertainty decomposition from: {best}')
_ = plot_epistemic_vs_aleatoric(results[best]['model'], data, n_grid=120)

**Prediction vs. truth on the simplex.** Left = true $E^*[Y\mid x]$, middle = MDN ensemble prediction, right = residual.

In [ ]:
_ = plot_prediction_vs_truth(results[best]['model'], data, n_grid=120)

**Per-method single-run summary** — NLL curve + labeled-composition scatter.

In [ ]:
for name, r in results.items():
    fig = make_single_run_figure(r, title=name)
    plt.show()